In [34]:
import pandas as pd
import numpy as np
import warnings, unicodedata
import os
import sys

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')


# ================= LIMPEZA =================

def _strip_accents(s):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))


def _clean_col(c):
    return _strip_accents(str(c)).replace('.', '').replace('%','').lower().strip()


def _coerce_pt_number(s):

    if pd.isna(s): 
        return np.nan

    s = str(s).strip().replace('.', '').replace(',', '.')

    try:
        return float(s)
    except:
        return np.nan


def _parse_pt_volume(v):

    if pd.isna(v):
        return np.nan

    s = str(v).strip()

    mult = 1

    if s.endswith(('M','m')):
        mult, s = 1e6, s[:-1]

    elif s.endswith(('K','k')):
        mult, s = 1e3, s[:-1]

    elif s.endswith(('B','b')):
        mult, s = 1e9, s[:-1]

    try:
        return float(s.replace('.', '').replace(',', '.')) * mult
    except:
        return np.nan


def standardize_investing_pt(df):

    cols = {_clean_col(c): c for c in df.columns}

    rename = {}

    for k,v in cols.items():

        if 'data' in k:
            rename[v] = 'Date'

        elif 'ultimo' in k or 'fechamento' in k:
            rename[v] = 'Close'

        elif 'abert' in k:
            rename[v] = 'Open'

        elif 'max' in k:
            rename[v] = 'High'

        elif 'min' in k:
            rename[v] = 'Low'

        elif 'vol' in k:
            rename[v] = 'Volume'


    out = df.rename(columns=rename).copy()

    out['Date'] = pd.to_datetime(out['Date'], dayfirst=True, errors='coerce')

    for c in ['Close','Open','High','Low']:

        if c in out:
            out[c] = out[c].apply(_coerce_pt_number)

    if 'Volume' in out:
        out['Volume'] = out['Volume'].apply(_parse_pt_volume)

    out = out.dropna(subset=['Date']).sort_values('Date').set_index('Date')

    return out


# ================= INDICADORES =================

def rsi(prices, period=14):

    d = prices.diff()

    up = d.clip(lower=0).rolling(period).mean()

    dn = (-d.clip(upper=0)).rolling(period).mean()

    rs = up / dn

    return 100 - (100 / (1 + rs))


def macd(prices, fast=12, slow=26):

    ema_f = prices.ewm(span=fast).mean()

    ema_s = prices.ewm(span=slow).mean()

    return ema_f - ema_s


def bollinger(close, win=20, mult=2):

    mid = close.rolling(win).mean()

    std = close.rolling(win).std()

    up = mid + mult * std

    low = mid - mult * std

    pctB = (close - low) / (up - low)

    return up, mid, low, pctB


def stoch_k(df, win=14):

    ll = df['Low'].rolling(win).min()

    hh = df['High'].rolling(win).max()

    return 100 * (df['Close'] - ll) / (hh - ll)


# ================= MODELO =================

class FinalModel:


    def __init__(self, csv_path, train_start_date, target_threshold, balance=True):

        self.csv_path = csv_path

        self.train_start_date = pd.to_datetime(train_start_date)

        self.target_threshold = target_threshold

        self.balance = balance

        self.scaler = StandardScaler()


    def load(self):

        raw = pd.read_csv(self.csv_path)

        self.df = standardize_investing_pt(raw)


    # ================= FEATURES =================

    def make_features(self):

        df = self.df.copy()

        df['Next_Close'] = df['Close'].shift(-1)

        df['Retorno_Pct'] = df['Close'].pct_change().shift(-1)

        df['Target'] = (df['Retorno_Pct'] > self.target_threshold).astype(int)


        df['Ret'] = df['Close'].pct_change() * 100

        for k in [1,2,3,5,10]:
            df[f'Ret_lag_{k}'] = df['Ret'].shift(k)


        df['SMA_5'] = df['Close'].rolling(5).mean()
        df['SMA_10'] = df['Close'].rolling(10).mean()
        df['SMA_20'] = df['Close'].rolling(20).mean()

        df['EMA_10'] = df['Close'].ewm(span=10).mean()
        df['EMA_20'] = df['Close'].ewm(span=20).mean()

        df["Dist_SMA5"] = (df["Close"] - df["SMA_5"]) / df["SMA_5"]
        df["Dist_SMA20"] = (df["Close"] - df["SMA_20"]) / df["SMA_20"]

        df["Momentum_3"] = df["Close"] - df["Close"].shift(3)
        df["Momentum_7"] = df["Close"] - df["Close"].shift(7)
        df["Momentum_14"] = df["Close"] - df["Close"].shift(14)

        df["Vol_5"] = df["Close"].pct_change().rolling(5).std()
        df["Vol_10"] = df["Close"].pct_change().rolling(10).std()
        df["Vol_20"] = df["Close"].pct_change().rolling(20).std()

        df['RSI_14'] = rsi(df['Close'])
        df['MACD'] = macd(df['Close'])

        _, _, _, pctB = bollinger(df['Close'])
        df['BB_pctB'] = pctB

        df['StochK'] = stoch_k(df)

        df["Trend"] = (df["SMA_5"] > df["SMA_20"]).astype(int)

        df["Range"] = df["High"] - df["Low"]
        df["Body"] = df["Close"] - df["Open"]

        df["dow"] = df.index.dayofweek
        df["month"] = df.index.month

        df = df.dropna()

        self.df_feat = df


    # ================= SPLIT =================

    def split_scale(self, test_size=200):

        df = self.df_feat

        X = df.drop(columns=['Open','High','Low','Close','Next_Close','Target','Retorno_Pct'], errors='ignore')

        X = X.select_dtypes(exclude='object')

        y = df['Target']


        X_train = X.iloc[:-test_size]
        y_train = y.iloc[:-test_size]

        X_test = X.iloc[-test_size:]
        y_test = y.iloc[-test_size:]


        X_train = X_train[X_train.index >= self.train_start_date]
        y_train = y_train[y_train.index >= self.train_start_date]


        self.X_train, self.y_train = X_train, y_train
        self.X_test, self.y_test = X_test, y_test


        self.scaler = StandardScaler().fit(self.X_train)

        self.X_train = pd.DataFrame(
            self.scaler.transform(self.X_train),
            index=self.X_train.index,
            columns=self.X_train.columns
        )

        self.X_test = pd.DataFrame(
            self.scaler.transform(self.X_test),
            index=self.X_test.index,
            columns=self.X_test.columns
        )


    # ================= MODELOS =================

    def fit_and_eval(self):

        models = {

            "RandomForest": RandomForestClassifier(
                n_estimators=1500,
                max_depth=12,
                min_samples_split=4,
                min_samples_leaf=1,
                max_features=0.7,
                max_samples=0.8,
                bootstrap=True,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            ),

            "XGBoost": XGBClassifier(
                n_estimators=800,
                max_depth=5,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                gamma=0.1,
                reg_lambda=1,
                eval_metric="logloss",
                random_state=42
            ),

            "SVM": SVC(
                kernel="rbf",
                C=20,
                gamma=0.05,
                probability=True,
                class_weight="balanced",
                random_state=42
            )
        }

        results = {}
        trained = {}

        for name, model in models.items():

            model.fit(self.X_train, self.y_train)

            trained[name] = model

            p = model.predict_proba(self.X_test)[:,1]

            y_pred = (p >= 0.45).astype(int)

            acc = accuracy_score(self.y_test, y_pred)

            hits = (self.y_test == y_pred).sum()

            results[name] = (acc*100, hits, len(self.y_test))

            print(f"{name}: {acc*100:.2f}% | {hits}/{len(self.y_test)}")


        best = max(results.items(), key=lambda x: x[1][0])[0]

        self.best_model = trained[best]

        print("\n✅ Melhor modelo:", best)


        p_best = self.best_model.predict_proba(self.X_test)[:,1]

        y_pred_best = (p_best >= 0.45).astype(int)

        print("\n===== CLASSIFICATION REPORT =====\n")

        print(classification_report(self.y_test, y_pred_best))

        return results


# ================= EXEC =================

if __name__ == "__main__":

    file_path = "Dados Históricos - Ibovespa 2006.csv"

    pipe = FinalModel(
        csv_path=file_path,
        train_start_date="2020-01-01",
        target_threshold=0.003,
        balance=True
    )

    pipe.load()

    pipe.make_features()

    pipe.split_scale(test_size=30)

    pipe.fit_and_eval()

RandomForest: 73.33% | 22/30
XGBoost: 80.00% | 24/30
SVM: 63.33% | 19/30

✅ Melhor modelo: XGBoost

===== CLASSIFICATION REPORT =====

              precision    recall  f1-score   support

           0       0.75      0.94      0.83        16
           1       0.90      0.64      0.75        14

    accuracy                           0.80        30
   macro avg       0.82      0.79      0.79        30
weighted avg       0.82      0.80      0.79        30

